In [ ]:
import altair as alt
import pandas as pd
import itertools

In [28]:
names = pd.read_csv("dpt2020.csv", sep=";")
names = names[names.preusuel != '_PRENOMS_RARES']
names = names[names.dpt != 'XX']
names = names[names.annais != 'XXXX']
names.annais = names.annais.astype(int)
names.nombre = names.nombre.astype(int)

names.sort_values('annais')

,sexe,preusuel,annais,dpt,nombre
1975231,2,AUGUSTA,1900,04,10
1483114,1,SAINTE-CROIX,1900,972,3
1483195,1,SAINTE-ROSE,1900,972,3
2466307,2,FRANCE,1900,974,3
2466306,2,FRANCE,1900,78,5
...,...,...,...,...,...
2273515,2,DJENA,2020,69,3
2273516,2,DJENA,2020,91,5
2273517,2,DJENA,2020,92,4
1211754,1,MIGUEL,2020,91,6


In [29]:
grouped_names = names.groupby(['preusuel', 'annais'], as_index=False)['nombre'].sum()
grouped_names.sort_values('annais')

,preusuel,annais,nombre
121395,JULIENNE,1900,870
35024,BONNE,1900,5
146830,MADELEINE,1900,4956
16394,ANGEL,1900,8
35028,BONNET,1900,7
...,...,...,...
216784,SHANON,2020,3
72962,EÏDEN,2020,6
72959,EZÉKIEL,2020,10
72954,EZÉCHIEL,2020,10


In [32]:
top = grouped_names.groupby('preusuel')['nombre'].sum().nlargest(20).index
filtered_names = grouped_names[grouped_names.preusuel.isin(top)]
filtered_names.sort_values('annais')

,preusuel,annais,nombre
6268,ALAIN,1900,83
204138,ROGER,1900,2060
85344,GEORGES,1900,5653
194643,PIERRE,1900,7461
109800,JEAN,1900,14100
...,...,...,...
192171,PAUL,2020,2468
194263,PHILIPPE,2020,58
203357,ROBERT,2020,17
15868,ANDRÉ,2020,52


In [ ]:
# fill the (name x year) grid with 0 so areas don't skip years
years = range(filtered_names['annais'].min(), filtered_names['annais'].max() + 1)
grid = pd.DataFrame(itertools.product(top, years), columns=['preusuel', 'annais'])
plot = grid.merge(plot, on=['preusuel', 'annais'], how='left').fillna({'nombre': 0})


chart = (
    alt.Chart(filtered_names)
    .mark_area(interpolate='monotone')
    .encode(
        x=alt.X('annais:Q', title='Année', axis=alt.Axis(format='d')),
        y=alt.Y('nombre:Q', stack='center', title=None, axis=None),  # streamgraph
        color=alt.Color('preusuel:N', title='Prénom',
                        scale=alt.Scale(scheme='tableau10')),
        # opacity=alt.condition(sel, alt.value(0.9), alt.value(0.15)),
        tooltip=['preusuel:N', 'annais:Q', 'nombre:Q'],
    )
    # .add_params(sel)
    .properties(width=800, height=400)
)
chart

alt.Chart(...)